# FY26 Q2 Demand Forecast — Correlation-Adjusted Time Series Model
**Cisco Product Portfolio | 30 Products | Professional Forecasting Framework**

---

## About This Notebook

This notebook implements a **complete, production-grade forecasting pipeline** for FY26 Q2 unit demand across 30 Cisco products. Every step mirrors industry practice: data validation before modelling, explicit assumptions, backtested model selection, and correlation-adjusted final forecasts.

**Pipeline overview:**

```
Raw Data
  └─ Step 1: Ingest & Validate
       └─ Step 2: Big Deal Adjustment  →  Adjusted Actuals
            └─ Step 3: EDA & Product Profiling
                 └─ Step 4: Correlation Analysis (Big Deal + Avg Deal)
                      └─ Step 5: Model Selection & Backtesting
                           └─ Step 6: Base Forecast (FY26 Q2)
                                └─ Step 7: Correlation Adjustment
                                     └─ Step 8: Final Forecast + Export
```

| Attribute | Detail |
|-----------|--------|
| Target quarter | FY26 Q2 (February – April 2026) |
| History | 12 quarters: FY23 Q2 → FY26 Q1 |
| Products | 30 |
| Models evaluated | WMA · SES · Holt's Linear · Linear Regression |
| Correlation layers | Big Deal co-movement · Avg Deal structural correlation |
| Accuracy metric | MAPE, MAE, Bias |


---
## Step 1 — Environment & Data Ingestion

### 1.1 Setup

All libraries, constants, and plot styles are declared once here. Changing `DATA_FILE` is the only edit needed to run this on an updated data pack.


In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import warnings
import pickle
import itertools
warnings.filterwarnings('ignore')

# ── Data ──────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr

# ── Modelling ─────────────────────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patches as mpatches

# ── Export ────────────────────────────────────────────────────────────────────
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── Reproducibility ───────────────────────────────────────────────────────────
np.random.seed(42)

# ─────────────────────────────────────────────────────────────────────────────
# GLOBAL CONSTANTS — edit only this block when data changes
# ─────────────────────────────────────────────────────────────────────────────
DATA_FILE = 'CFL_External_Data_Pack_Phase1_JN.xlsx'

ACTUAL_QUARTERS = [
    'FY23 Q2','FY23 Q3','FY23 Q4',
    'FY24 Q1','FY24 Q2','FY24 Q3','FY24 Q4',
    'FY25 Q1','FY25 Q2','FY25 Q3','FY25 Q4',
    'FY26 Q1'
]
BD_QUARTERS = ['2024Q2','2024Q3','2024Q4','2025Q1',
               '2025Q2','2025Q3','2025Q4','2026Q1']
BD_TO_ACTUAL = {
    '2024Q2':'FY24 Q2','2024Q3':'FY24 Q3','2024Q4':'FY24 Q4',
    '2025Q1':'FY25 Q1','2025Q2':'FY25 Q2','2025Q3':'FY25 Q3',
    '2025Q4':'FY25 Q4','2026Q1':'FY26 Q1'
}
OVERLAP_ACTUAL = [BD_TO_ACTUAL[q] for q in BD_QUARTERS]
TARGET         = 'FY26 Q2'
N_PRODUCTS     = 30

# ── Plot theme ────────────────────────────────────────────────────────────────
PALETTE = {
    'navy'  : '#1F4E79',
    'blue'  : '#2E75B6',
    'teal'  : '#0D7377',
    'green' : '#375623',
    'amber' : '#C07000',
    'red'   : '#C00000',
    'purple': '#7030A0',
    'grey'  : '#595959',
    'light' : '#F0F4FF',
}
plt.rcParams.update({
    'figure.facecolor' : 'white',
    'axes.facecolor'   : '#F8F9FA',
    'axes.grid'        : True,
    'grid.color'       : '#E0E0E0',
    'grid.linestyle'   : '--',
    'grid.alpha'       : 0.7,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.spines.left' : True,
    'axes.spines.bottom':True,
    'font.family'      : 'DejaVu Sans',
    'axes.labelsize'   : 10,
    'axes.titlesize'   : 11,
    'xtick.labelsize'  : 8,
    'ytick.labelsize'  : 8,
})

print('✅  Environment ready')
print(f'    Target: {TARGET} | {N_PRODUCTS} products | {len(ACTUAL_QUARTERS)} quarters of history')

### 1.2 Data Ingestion

Each sheet is parsed defensively: no assumed header rows, explicit column mapping, and forced numeric casting. A spot-check cell follows to confirm alignment before any modelling begins.


In [ ]:
xl = pd.ExcelFile(DATA_FILE)
print('Sheets found:', xl.sheet_names)

# ── Actual Bookings ───────────────────────────────────────────────────────────
raw_act = pd.read_excel(xl, sheet_name='Data Pack - Actual Bookings', header=None)
actuals = raw_act.iloc[3:33].copy()
actuals.columns = (
    ['Cost_Rank','Product_Name','Life_Cycle'] +
    ACTUAL_QUARTERS +
    ['Your_Forecast','Demand_Planner','Marketing','Data_Science'] +
    ['_x1','_x2','_x3']
)
keep_cols = ['Cost_Rank','Product_Name','Life_Cycle'] + ACTUAL_QUARTERS + \
            ['Demand_Planner','Marketing','Data_Science']
actuals = actuals[keep_cols].reset_index(drop=True)
actuals['Cost_Rank'] = pd.to_numeric(actuals['Cost_Rank'], errors='coerce')
for q in ACTUAL_QUARTERS:
    actuals[q] = pd.to_numeric(actuals[q], errors='coerce')

# ── Big Deal (robust column detection) ───────────────────────────────────────
raw_bd  = pd.read_excel(xl, sheet_name='Big Deal', header=None)
row0    = raw_bd.iloc[0].tolist()
tc = next(i for i,v in enumerate(row0) if str(v)=='MFG Book Units')
bc = next(i for i,v in enumerate(row0) if str(v)=='Big Deals')
ac = next(i for i,v in enumerate(row0) if str(v)=='Avg Deals')

bd_df = raw_bd.iloc[2:32, list(range(0,2))+list(range(tc,bc))+list(range(bc,ac))+list(range(ac,ac+len(BD_QUARTERS)))].copy()
bd_df.columns = (
    ['Cost_Rank','Product_Name'] +
    [f'Total_{q}' for q in BD_QUARTERS] +
    [f'BigDeal_{q}' for q in BD_QUARTERS] +
    [f'AvgDeal_{q}' for q in BD_QUARTERS]
)
bd_df = bd_df.dropna(subset=['Cost_Rank']).reset_index(drop=True)
for col in bd_df.columns[2:]:
    bd_df[col] = pd.to_numeric(bd_df[col], errors='coerce').fillna(0)

# ── SCMS ──────────────────────────────────────────────────────────────────────
SCMS_QS  = ['2023Q1','2023Q2','2023Q3','2023Q4',
             '2024Q1','2024Q2','2024Q3','2024Q4',
             '2025Q1','2025Q2','2025Q3','2025Q4','2026Q1']
raw_scms = pd.read_excel(xl, sheet_name='SCMS', header=None)
scms_df  = raw_scms.iloc[3:].copy()
scms_df.columns = ['Cost_Rank','Product_Name','SCMS'] + SCMS_QS
scms_df = scms_df.dropna(subset=['Cost_Rank']).reset_index(drop=True)
scms_df['Cost_Rank'] = pd.to_numeric(scms_df['Cost_Rank'], errors='coerce')
for q in SCMS_QS:
    scms_df[q] = pd.to_numeric(scms_df[q], errors='coerce')

# ── VMS ───────────────────────────────────────────────────────────────────────
raw_vms = pd.read_excel(xl, sheet_name='VMS', header=None)
vms_df  = raw_vms.iloc[3:].copy()
vms_df.columns = ['Cost_Rank','Product_Name','VMS'] + SCMS_QS
vms_df  = vms_df.dropna(subset=['Cost_Rank']).reset_index(drop=True)
vms_df['Cost_Rank'] = pd.to_numeric(vms_df['Cost_Rank'], errors='coerce')

print(f'\nActuals : {actuals.shape[0]} products × {len(ACTUAL_QUARTERS)} quarters')
print(f'Big Deal: {bd_df.shape[0]} products × {len(BD_QUARTERS)} quarters per section')
print(f'SCMS    : {scms_df["SCMS"].nunique()} segments, {scms_df.shape[0]} rows')
print(f'VMS     : {vms_df["VMS"].nunique()} verticals, {vms_df.shape[0]} rows')

In [ ]:
# ── 1.3 Spot-check — Verify data alignment before proceeding ──────────────────
# Known values from the source file:
CHECKS = [
    (0, 'FY24 Q2', 9437,  'Product #1 FY24 Q2'),
    (0, 'FY26 Q1', 13144, 'Product #1 FY26 Q1'),
    (1, 'FY24 Q2', 59424, 'Product #2 FY24 Q2'),
]
all_pass = True
print('Data alignment checks:')
for idx, q, expected, label in CHECKS:
    actual_val = int(actuals.iloc[idx][q])
    status = '✅ PASS' if actual_val == expected else '❌ FAIL'
    if actual_val != expected:
        all_pass = False
    print(f'  {status}  {label}: got {actual_val:,} (expected {expected:,})')

print()
if all_pass:
    print('All checks passed — data correctly aligned. Proceeding.')
else:
    print('⚠️  One or more checks failed. Review column mapping before proceeding.')

---
## Step 2 — Big Deal Adjustment

**What is a Big Deal?**  
A Big Deal is a single large customer order that inflates bookings in the quarter it lands. It is irregular, non-recurring (or at best periodic), and completely unrelated to the underlying demand trend.

**Why remove it before forecasting?**  
A model trained on a series that contains a spike it cannot explain will try to reproduce that spike in the forecast — causing systematic over-forecasting. The correct approach is to:
1. Train the base model on *Avg Deal* (regular recurring demand).
2. Forecast Big Deals *separately* using historical frequency and sales pipeline data.
3. Combine at the end: `Final Forecast = Base Forecast + Big Deal Overlay`.

**Formula:**  
`Avg Deal Units = Raw Bookings − Big Deal Units`


In [ ]:
# ── 2.1 Build Avg-Deal-adjusted actuals ───────────────────────────────────────
adjusted = actuals.copy()
for bd_q, act_q in BD_TO_ACTUAL.items():
    adjusted[act_q] = (
        pd.to_numeric(adjusted[act_q], errors='coerce') -
        bd_df[f'BigDeal_{bd_q}'].values
    )

# ── 2.2 Big Deal impact summary ───────────────────────────────────────────────
bd_summary = []
for i, row in actuals.iterrows():
    bd_row   = bd_df.iloc[i]
    tot_raw  = sum(pd.to_numeric(row[q], errors='coerce') or 0 for q in OVERLAP_ACTUAL)
    tot_bd   = sum(bd_row[f'BigDeal_{q}'] for q in BD_QUARTERS)
    avg_pct  = tot_bd / tot_raw if tot_raw > 0 else 0
    n_spikes = sum(1 for q in BD_QUARTERS
                   if bd_row[f'BigDeal_{q}'] / (bd_row[f'Total_{q}'] + 1e-9) > 0.15)
    tier = 'HIGH' if avg_pct > 0.15 else ('MEDIUM' if avg_pct >= 0.05 else 'LOW')
    bd_summary.append({
        'Rank':int(row['Cost_Rank']), 'Product':row['Product_Name'],
        'Life_Cycle':row['Life_Cycle'],
        'BD_Units':int(tot_bd), 'Raw_Units':int(tot_raw),
        'BD_Pct':round(avg_pct*100,1), 'Spike_Qtrs':n_spikes, 'Tier':tier
    })
bd_sum = pd.DataFrame(bd_summary)

print('Big Deal Impact Summary — All 30 Products')
print('─'*80)
print(f'{"#":<4}{"Product":<46}{"Life Cycle":<13}{"BD %":>6}{"Tier":>10}')
print('─'*80)
for _,r in bd_sum.iterrows():
    flag = '  ⚠' if r['Tier']=='HIGH' else ''
    print(f"{int(r['Rank']):<4}{str(r['Product'])[:44]:<46}{r['Life_Cycle']:<13}{r['BD_Pct']:>5.1f}%{r['Tier']:>10}{flag}")

print('\nTier distribution:')
print(bd_sum['Tier'].value_counts().to_string())

In [ ]:
# ── 2.3 Visualise Big Deal impact ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bar chart — BD% by product
colors = [PALETTE['red'] if t=='HIGH' else PALETTE['amber'] if t=='MEDIUM'
          else PALETTE['teal'] for t in bd_sum['Tier']]
axes[0].barh(bd_sum['Rank'][::-1], bd_sum['BD_Pct'][::-1],
             color=list(reversed(colors)), height=0.7, edgecolor='white')
axes[0].axvline(15, color=PALETTE['red'],   linestyle='--', lw=1.2, label='HIGH (>15%)')
axes[0].axvline(5,  color=PALETTE['amber'], linestyle='--', lw=1.2, label='MEDIUM (>5%)')
axes[0].set_yticks(bd_sum['Rank'][::-1])
axes[0].set_yticklabels([f"#{r}" for r in bd_sum['Rank'][::-1]], fontsize=7)
axes[0].set_xlabel('Big Deal % of Total Sales (Last 8 Quarters)')
axes[0].set_title('Big Deal Exposure by Product', fontweight='bold')
axes[0].legend(fontsize=8)

# Before / After — Product #2 (highest exposure)
p2_raw  = pd.to_numeric(actuals.iloc[1][ACTUAL_QUARTERS], errors='coerce').values
p2_adj  = pd.to_numeric(adjusted.iloc[1][ACTUAL_QUARTERS], errors='coerce').values
x       = range(len(ACTUAL_QUARTERS))
axes[1].plot(x, p2_raw, 'o-', color=PALETTE['red'],   lw=2,   ms=5, label='Raw Actuals')
axes[1].plot(x, p2_adj, 's--',color=PALETTE['teal'],  lw=2,   ms=5, label='Avg Deal (adjusted)')
axes[1].fill_between(x, p2_adj, p2_raw, alpha=0.15, color=PALETTE['amber'], label='Big Deal volume')
axes[1].set_xticks(range(len(ACTUAL_QUARTERS)))
axes[1].set_xticklabels(ACTUAL_QUARTERS, rotation=40, ha='right')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:,.0f}'))
axes[1].set_title('Product #2 — WiFi6E Indoor: Raw vs Adjusted', fontweight='bold')
axes[1].legend(fontsize=9)

plt.suptitle('Step 2 — Big Deal Adjustment', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Step 3 — Exploratory Data Analysis

EDA answers three questions before any model is selected:
1. **Trend** — Is the product growing, declining, or flat over the observation period?
2. **Volatility** — How much does demand swing quarter to quarter? (Coefficient of Variation)
3. **Completeness** — How many quarters of usable history does each product have?

These three attributes determine which forecasting model is appropriate per product.


In [ ]:
# ── 3.1 Compute EDA metrics per product ───────────────────────────────────────
eda_rows = []
for i, row in adjusted.iterrows():
    s = pd.to_numeric(row[ACTUAL_QUARTERS], errors='coerce').dropna().values
    if len(s) < 2:
        continue
    mean_v = s.mean()
    std_v  = s.std(ddof=1)
    cv     = std_v / mean_v if mean_v > 0 else 0

    # Linear slope via OLS — units per quarter
    x       = np.arange(len(s)).reshape(-1,1)
    slope   = LinearRegression().fit(x, s).coef_[0]
    trend_l = 'Growing' if slope > 200 else ('Declining' if slope < -200 else 'Stable')

    # Quarter-of-year seasonal check — which Cisco FY quarter is strongest?
    slot_vals = {0:[],1:[],2:[],3:[]}
    for qi, q in enumerate(ACTUAL_QUARTERS):
        v = pd.to_numeric(row[q], errors='coerce')
        if not np.isnan(v):
            slot_vals[qi % 4].append(v)
    slot_means = {k:np.mean(v) for k,v in slot_vals.items() if v}
    peak_slot  = max(slot_means, key=slot_means.get) if slot_means else None
    seas_ratio = (max(slot_means.values())/min(slot_means.values())
                  if len(slot_means)>1 and min(slot_means.values())>0 else 1.0)
    slot_label = {0:'FY Q2',1:'FY Q3',2:'FY Q4',3:'FY Q1'}

    eda_rows.append({
        'Rank'       : int(row['Cost_Rank']),
        'Product'    : row['Product_Name'],
        'Life_Cycle' : row['Life_Cycle'],
        'N_Qtrs'     : len(s),
        'Mean'       : round(mean_v),
        'Std'        : round(std_v),
        'CV'         : round(cv, 3),
        'Min'        : int(s.min()),
        'Max'        : int(s.max()),
        'Trend'      : trend_l,
        'Slope_per_Q': round(slope, 1),
        'Peak_Slot'  : slot_label.get(peak_slot,'n/a'),
        'Seas_Ratio' : round(seas_ratio, 3),
    })

eda = pd.DataFrame(eda_rows)

print('EDA Summary — 30 Products (Avg Deal adjusted series)')
print('─'*105)
print(f'{"#":<4}{"Product":<46}{"LC":<12}{"N":>3}{"Mean":>9}{"CV":>7}{"Trend":<12}{"Slope/Q":>9}{"Seas R":>8}')
print('─'*105)
for _,r in eda.iterrows():
    print(f"{int(r['Rank']):<4}{str(r['Product'])[:44]:<46}{r['Life_Cycle']:<12}"
          f"{r['N_Qtrs']:>3}{int(r['Mean']):>9,}{r['CV']:>7.3f}"
          f"  {r['Trend']:<10}{r['Slope_per_Q']:>9.1f}{r['Seas_Ratio']:>8.2f}")

In [ ]:
# ── 3.2 EDA Visualisation ─────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# Plot 1: CV distribution
ax1 = fig.add_subplot(gs[0, 0])
cv_colors = [PALETTE['red'] if cv>0.30 else PALETTE['amber'] if cv>0.15
             else PALETTE['teal'] for cv in eda['CV']]
ax1.bar(eda['Rank'], eda['CV'], color=cv_colors, edgecolor='white')
ax1.axhline(0.30, color=PALETTE['red'],   ls='--', lw=1.2, label='High (>0.30)')
ax1.axhline(0.15, color=PALETTE['amber'], ls='--', lw=1.2, label='Med (>0.15)')
ax1.set_title('Coefficient of Variation (CV)', fontweight='bold')
ax1.set_xlabel('Product Rank')
ax1.set_ylabel('CV')
ax1.legend(fontsize=7)

# Plot 2: Trend distribution pie
ax2 = fig.add_subplot(gs[0, 1])
tc  = eda['Trend'].value_counts()
ax2.pie(tc, labels=tc.index, autopct='%1.0f%%',
        colors=[PALETTE['teal'],PALETTE['red'],PALETTE['blue']],
        startangle=90, textprops={'fontsize':10})
ax2.set_title('Trend Distribution', fontweight='bold')

# Plot 3: Life Cycle pie
ax3 = fig.add_subplot(gs[0, 2])
lc  = eda['Life_Cycle'].value_counts()
ax3.pie(lc, labels=lc.index, autopct='%1.0f%%',
        colors=[PALETTE['navy'],PALETTE['red'],PALETTE['green']],
        startangle=90, textprops={'fontsize':10})
ax3.set_title('Product Life Cycle Mix', fontweight='bold')

# Plots 4–6: Series for 3 contrasting products
show_ranks = [1, 5, 22]
lc_colors  = {'Sustaining':PALETTE['navy'],'Decline':PALETTE['red'],'NPI-Ramp':PALETTE['green']}
for ax_idx, rank in enumerate(show_ranks):
    ax  = fig.add_subplot(gs[1, ax_idx])
    row = adjusted[adjusted['Cost_Rank']==rank].iloc[0]
    ev  = eda[eda['Rank']==rank].iloc[0]
    s   = pd.to_numeric(row[ACTUAL_QUARTERS], errors='coerce').values
    col = lc_colors.get(row['Life_Cycle'], PALETTE['grey'])
    ax.plot(range(len(ACTUAL_QUARTERS)), s, 'o-', color=col, lw=2, ms=5)
    ax.fill_between(range(len(ACTUAL_QUARTERS)), s, alpha=0.08, color=col)

    # Trend line
    mask = ~np.isnan(s)
    if mask.sum() > 1:
        xm  = np.where(mask)[0]
        coef = np.polyfit(xm, s[mask], 1)
        ax.plot(xm, np.polyval(coef, xm), '--', color=PALETTE['amber'], lw=1.2, alpha=0.7, label='Trend')

    ax.set_xticks(range(len(ACTUAL_QUARTERS)))
    ax.set_xticklabels(ACTUAL_QUARTERS, rotation=40, ha='right', fontsize=6)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{int(v):,}'))
    ax.set_title(f"#{rank} {str(row['Product_Name'])[:28]}\n"
                 f"{row['Life_Cycle']} | CV={ev['CV']:.2f} | {ev['Trend']}",
                 fontsize=8, fontweight='bold')
    if ax_idx==0: ax.legend(fontsize=7)

plt.suptitle('Step 3 — Exploratory Data Analysis', fontsize=13, fontweight='bold', y=1.01)
plt.savefig('eda_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 EDA saved to eda_summary.png')

---
## Step 4 — Correlation Analysis

Correlation between products reveals **demand linkages** — products that consistently rise and fall together. We analyse two separate series:

| Series | What It Captures | Why It Matters |
|--------|-----------------|----------------|
| **Big Deal** | One-off large order co-movement | Same customer buying multiple product types at once |
| **Avg Deal** | Regular recurring demand trend | Structural market demand — most reliable for forecasting |

Avg Deal correlations are far more important for forecasting because they represent stable, repeating demand patterns that a model can and should incorporate.

**Interpretation thresholds:**
- `|r| ≥ 0.85` — Very strong: treat as a demand cluster; use partner trend to validate forecast direction
- `0.70 ≤ |r| < 0.85` — Moderate: use as a soft cross-check
- `r < 0` — Negative: products compete for the same budget; a BD spike in one reduces probability of the other


In [ ]:
# ── 4.1 Correlation matrix builder ────────────────────────────────────────────
def compute_corr_matrix(data_matrix, names, ranks, threshold=0.60, p_threshold=0.10):
    """
    Compute Pearson and Spearman correlation matrices.
    Returns the full matrix and a filtered list of strong pairs.
    """
    n = data_matrix.shape[0]
    P = np.ones((n, n))
    S = np.ones((n, n))
    PV= np.ones((n, n))

    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            try:
                r, p   = pearsonr(data_matrix[i], data_matrix[j])
                sr, _  = spearmanr(data_matrix[i], data_matrix[j])
                P[i,j]  = r
                S[i,j]  = sr
                PV[i,j] = p
            except Exception:
                P[i,j] = np.nan

    pairs = []
    for i in range(n):
        for j in range(i+1, n):
            r  = P[i,j]
            p  = PV[i,j]
            sr = S[i,j]
            if not np.isnan(r) and abs(r) >= threshold and p <= p_threshold:
                pairs.append({
                    'i':i, 'j':j,
                    'rank_i':ranks[i], 'rank_j':ranks[j],
                    'name_i':names[i], 'name_j':names[j],
                    'pearson_r':round(r,3),
                    'spearman_r':round(sr,3),
                    'p_value':round(p,4),
                    'direction':'Positive' if r>0 else 'Negative',
                    'strength':'Very Strong' if abs(r)>=0.85 else 'Strong'
                })
    pairs.sort(key=lambda x: -abs(x['pearson_r']))
    return P, S, PV, pairs


ranks = adjusted['Cost_Rank'].astype(int).tolist()
names = adjusted['Product_Name'].tolist()

# ── Big Deal correlation ───────────────────────────────────────────────────────
bd_matrix  = bd_df[[f'BigDeal_{q}' for q in BD_QUARTERS]].values.astype(float)
BD_P, BD_S, BD_PV, bd_pairs = compute_corr_matrix(bd_matrix, names, ranks)

# ── Avg Deal correlation ───────────────────────────────────────────────────────
avg_matrix = bd_df[[f'AvgDeal_{q}' for q in BD_QUARTERS]].values.astype(float)
AVG_P, AVG_S, AVG_PV, avg_pairs = compute_corr_matrix(avg_matrix, names, ranks)

print(f'Big Deal  — strong pairs (|r|>0.60, p<0.10): {len(bd_pairs)}')
print(f'Avg Deal  — strong pairs (|r|>0.60, p<0.10): {len(avg_pairs)}')
print(f'\nAvg Deal — Very Strong pairs (|r|≥0.85): {sum(1 for p in avg_pairs if abs(p["pearson_r"])>=0.85)}')
print(f'Avg Deal — Negative correlations         : {sum(1 for p in avg_pairs if p["pearson_r"]<0)}')

In [ ]:
# ── 4.2 Print top correlations — Big Deal ─────────────────────────────────────
print('TOP 10 BIG DEAL CORRELATIONS')
print('─'*90)
print(f'{"Rank A":<8}{"Product A":<42}{"Rank B":<8}{"Product B":<42}{"r":>7}{"Dir":<12}')
print('─'*90)
for p in bd_pairs[:10]:
    print(f"#{p['rank_i']:<7}{str(p['name_i'])[:40]:<42}#{p['rank_j']:<7}"
          f"{str(p['name_j'])[:40]:<42}{p['pearson_r']:>7.3f}  {p['direction']}")

In [ ]:
# ── 4.3 Print top correlations — Avg Deal ─────────────────────────────────────
print('TOP 20 AVG DEAL CORRELATIONS (Regular Demand — used for forecast adjustment)')
print('─'*90)
print(f'{"Rank A":<8}{"Product A":<42}{"Rank B":<8}{"Product B":<42}{"r":>7}{"Strength":<14}')
print('─'*90)
for p in avg_pairs[:20]:
    marker = '🔴' if p['pearson_r'] < 0 else '🟢'
    print(f"#{p['rank_i']:<7}{str(p['name_i'])[:40]:<42}#{p['rank_j']:<7}"
          f"{str(p['name_j'])[:40]:<42}{p['pearson_r']:>7.3f}  {marker} {p['strength']}")

In [ ]:
# ── 4.4 Correlation heatmaps — Big Deal vs Avg Deal ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

cmap_div = LinearSegmentedColormap.from_list(
    'rg', ['#C00000','#FFFFFF','#1F4E79'], N=256
)

for ax, matrix, title in [
    (axes[0], BD_P,  'Big Deal Correlation'),
    (axes[1], AVG_P, 'Avg Deal Correlation (Structural Demand)')
]:
    m = np.where(np.isnan(matrix), 0, matrix)
    im = ax.imshow(m, cmap=cmap_div, vmin=-1, vmax=1, aspect='auto')
    ax.set_xticks(range(len(ranks)))
    ax.set_yticks(range(len(ranks)))
    ax.set_xticklabels([f'#{r}' for r in ranks], rotation=90, fontsize=6)
    ax.set_yticklabels([f'#{r}' for r in ranks], fontsize=6)
    ax.set_title(title, fontweight='bold', fontsize=11)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Pearson r')

plt.suptitle('Step 4 — Correlation Heatmaps: Big Deal vs Avg Deal',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Correlation heatmaps saved to correlation_heatmaps.png')
print('\n💡 Note: Avg Deal shows denser clustering → structural demand linkages')
print('   These pairs will be used in Step 7 to cross-validate and adjust the base forecast.')

In [ ]:
# ── 4.5 Build demand clusters from Avg Deal correlations ──────────────────────
# Cluster = group of products with very strong mutual Avg Deal correlations (r >= 0.85)
# These clusters are used in Step 7 to apply group-level directional adjustments.

from collections import defaultdict

very_strong = [(p['rank_i'], p['rank_j'], p['pearson_r'])
               for p in avg_pairs if abs(p['pearson_r']) >= 0.85 and p['pearson_r'] > 0]

# Union-Find to identify connected components
parent = {r: r for r in ranks}
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
def union(a, b):
    parent[find(a)] = find(b)

for ri, rj, _ in very_strong:
    union(ri, rj)

clusters = defaultdict(list)
for r in ranks:
    clusters[find(r)].append(r)

CLUSTERS = {k: sorted(v) for k, v in clusters.items() if len(v) > 1}
singletons = [v[0] for v in clusters.values() if len(v) == 1]

print(f'Avg Deal demand clusters (|r| ≥ 0.85, positive):')
print(f'  Multi-product clusters: {len(CLUSTERS)}')
print(f'  Standalone products   : {len(singletons)}')
print()
for ci, (key, members) in enumerate(CLUSTERS.items(), 1):
    mnames = [str(adjusted[adjusted['Cost_Rank']==m]['Product_Name'].values[0])[:35]
              for m in members]
    print(f'  Cluster {ci}: {members}')
    for mn in mnames:
        print(f'            → {mn}')
    print()

---
## Step 5 — Model Building & Backtesting

### 5.1 Model Definitions

Four models are implemented from first principles. Building from scratch (rather than calling library functions) ensures full understanding of what each model computes and why it may fail.

| Model | Logic | Best For |
|-------|-------|----------|
| **WMA** | Linearly weights last N quarters; recent = higher weight | Volatile, no clear trend |
| **SES** | Exponentially decays older values; one tunable parameter α | NPI-Ramp, short series |
| **Holt's Linear** | Tracks level AND trend; two parameters α, β | Sustaining with trend |
| **Linear Regression** | OLS best-fit line extrapolated one step | Decline / strong trend |


In [ ]:
# ── 5.1 Model implementations ─────────────────────────────────────────────────

def wma(series, n=4):
    """Weighted Moving Average — linearly increasing weights over last n periods."""
    s = np.asarray(series, dtype=float)
    s = s[~np.isnan(s)]
    if len(s) == 0: return np.nan
    n = min(n, len(s))
    w = np.arange(1, n+1, dtype=float)
    return float(np.dot(w, s[-n:]) / w.sum())


def ses(series):
    """
    Simple Exponential Smoothing with optimised alpha (grid search).
    S_t = α·y_t + (1-α)·S_{t-1}; forecast = S_T (last smoothed level).
    """
    s = np.asarray(series, dtype=float)
    s = s[~np.isnan(s)]
    if len(s) == 0: return np.nan
    if len(s) == 1: return float(s[0])

    def _sse(a):
        lv = s[0]; e = 0.0
        for obs in s[1:]:
            e += (obs - lv)**2
            lv = a*obs + (1-a)*lv
        return e

    best_a = min(np.arange(0.05, 1.0, 0.05), key=_sse)
    lv = s[0]
    for obs in s[1:]:
        lv = best_a*obs + (1-best_a)*lv
    return float(lv)


def holts(series, h=1):
    """
    Holt's Double Exponential Smoothing (linear trend).
    Level : L_t = α·y_t + (1-α)(L_{t-1} + T_{t-1})
    Trend : T_t = β·(L_t - L_{t-1}) + (1-β)·T_{t-1}
    Forecast(h) = L_T + h·T_T
    Optimises α and β jointly via grid search.
    """
    s = np.asarray(series, dtype=float)
    s = s[~np.isnan(s)]
    if len(s) < 3: return ses(s)

    def _sse(a, b):
        lv = s[0]; tr = s[1] - s[0]; e = 0.0
        for obs in s[1:]:
            e   += (obs - (lv + tr))**2
            nl   = a*obs + (1-a)*(lv+tr)
            tr   = b*(nl-lv) + (1-b)*tr
            lv   = nl
        return e

    best, best_e = (0.3, 0.1), float('inf')
    for a in np.arange(0.1, 1.0, 0.15):
        for b in np.arange(0.05, 0.55, 0.10):
            e = _sse(a, b)
            if e < best_e:
                best_e, best = e, (a, b)

    a, b = best
    lv = s[0]; tr = s[1] - s[0]
    for obs in s[1:]:
        nl = a*obs + (1-a)*(lv+tr)
        tr = b*(nl-lv) + (1-b)*tr
        lv = nl
    return float(max(0, lv + h*tr))


def linreg(series, h=1):
    """
    OLS linear trend extrapolation: y = a + b·t, forecast at t = T + h.
    Floor at zero — sales cannot be negative.
    """
    s    = np.asarray(series, dtype=float)
    mask = ~np.isnan(s)
    x    = np.where(mask)[0].reshape(-1, 1)
    y    = s[mask]
    if len(y) < 2: return float(y[-1]) if len(y) > 0 else np.nan
    t_next = np.array([[len(s) + h - 1]])
    return float(max(0, LinearRegression().fit(x, y).predict(t_next)[0]))


def run_all_models(series):
    """Return forecasts from all four models as a dict."""
    return {
        'WMA'  : wma(series),
        'SES'  : ses(series),
        'Holt' : holts(series),
        'LR'   : linreg(series),
    }


print('✅ Model functions defined:')
print('   WMA  — Weighted Moving Average (last 4 quarters, linearly weighted)')
print('   SES  — Simple Exponential Smoothing (optimised α)')
print('   Holt — Holt Double Exponential (optimised α, β)')
print('   LR   — OLS Linear Regression (trend extrapolation)')

In [ ]:
# ── 5.2 Model assignment rules ─────────────────────────────────────────────────
# Rules applied in priority order:
#   1. NPI-Ramp or fewer than 6 quarters  →  SES
#   2. Decline lifecycle                  →  LR
#   3. CV > 0.30 (high volatility)        →  WMA
#   4. Otherwise (Sustaining, moderate)   →  Holt

def assign_model_rule(life_cycle, cv, n_qtrs):
    if life_cycle == 'NPI-Ramp' or n_qtrs < 6:
        return 'SES'
    if life_cycle == 'Decline':
        return 'LR'
    if cv > 0.30:
        return 'WMA'
    return 'Holt'

eda['Rule_Model'] = eda.apply(
    lambda r: assign_model_rule(r['Life_Cycle'], r['CV'], r['N_Qtrs']), axis=1
)

print('Model assignment by rule:')
print(eda['Rule_Model'].value_counts().to_string())
print()
print(f'{"#":<4}{"Product":<46}{"Life Cycle":<13}{"CV":>6}  {"Rule Model"}')
print('─'*80)
for _,r in eda.iterrows():
    print(f"{int(r['Rank']):<4}{str(r['Product'])[:44]:<46}{r['Life_Cycle']:<13}"
          f"{r['CV']:>6.3f}  {r['Rule_Model']}")

In [ ]:
# ── 5.3 Backtest — train on Q1..Q11, evaluate on Q12 (FY26 Q1) ────────────────
#
# Q12 (FY26 Q1) is the most recent actual we have. By withholding it,
# we simulate what a forecast made at the end of FY25 Q4 would have looked like.
# This is the only honest way to measure model accuracy before the target quarter lands.

TRAIN_QS   = ACTUAL_QUARTERS[:-1]   # FY23 Q2 → FY25 Q4
HOLDOUT_Q  = ACTUAL_QUARTERS[-1]    # FY26 Q1
MAPE = lambda f,a: abs(f-a)/a*100 if a>0 and not np.isnan(f) else np.nan
BIAS = lambda f,a: (f-a)/a         if a>0 and not np.isnan(f) else np.nan

backtest_rows = []

for i, row in adjusted.iterrows():
    train_s  = pd.to_numeric(row[TRAIN_QS],  errors='coerce').values
    actual_h = pd.to_numeric(row[HOLDOUT_Q], errors='coerce')
    if np.isnan(actual_h): continue

    preds    = run_all_models(train_s)
    eda_row  = eda[eda['Rank']==int(row['Cost_Rank'])].iloc[0]
    rule_mdl = eda_row['Rule_Model']

    mapes = {m: MAPE(preds[m], actual_h) for m in preds}

    # Best model = lowest backtest MAPE
    valid_mapes = {m:v for m,v in mapes.items() if not np.isnan(v)}
    best_model  = min(valid_mapes, key=valid_mapes.get) if valid_mapes else rule_mdl
    best_mape   = valid_mapes.get(best_model, np.nan)
    rule_mape   = mapes.get(rule_mdl, np.nan)

    # Override: only if backtest evidence beats rule by > 5 pp
    override = (not np.isnan(rule_mape) and not np.isnan(best_mape)
                and rule_mape - best_mape > 5.0
                and best_model != rule_mdl)
    final_model = best_model if override else rule_mdl
    final_mape  = mapes.get(final_model, np.nan)

    backtest_rows.append({
        'Rank'       : int(row['Cost_Rank']),
        'Product'    : row['Product_Name'],
        'Life_Cycle' : row['Life_Cycle'],
        'Actual_Q12' : actual_h,
        'Rule_Model' : rule_mdl,
        'Best_Model' : final_model,
        'Override'   : override,
        'F_WMA'      : preds['WMA'],
        'F_SES'      : preds['SES'],
        'F_Holt'     : preds['Holt'],
        'F_LR'       : preds['LR'],
        'MAPE_WMA'   : mapes['WMA'],
        'MAPE_SES'   : mapes['SES'],
        'MAPE_Holt'  : mapes['Holt'],
        'MAPE_LR'    : mapes['LR'],
        'MAPE_Final' : final_mape,
        'Bias_Final' : BIAS(preds.get(final_model, np.nan), actual_h),
    })

bt = pd.DataFrame(backtest_rows)

print(f'Backtest Results — Predicting {HOLDOUT_Q} using {TRAIN_QS[0]}→{TRAIN_QS[-1]}')
print('─'*100)
print(f'{"#":<4}{"Product":<42}{"Actual":>9}{"Best Model":<12}{"MAPE %":>8}{"Override"}')
print('─'*100)
for _,r in bt.iterrows():
    ov = '  ⚡ Override' if r['Override'] else ''
    print(f"{int(r['Rank']):<4}{str(r['Product'])[:40]:<42}"
          f"{int(r['Actual_Q12']):>9,}  {r['Best_Model']:<12}{r['MAPE_Final']:>7.1f}%{ov}")

print('─'*100)
print(f"\nPortfolio-level backtest accuracy:")
for m in ['WMA','SES','Holt','LR','Final']:
    col = f'MAPE_{m}'
    vals = bt[col].dropna()
    print(f"  {m:<8}: Mean MAPE = {vals.mean():.1f}%  Median = {vals.median():.1f}%  "
          f"Products <10% = {(vals<10).sum()}/{len(vals)}")

In [ ]:
# ── 5.4 Backtest visualisation ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: MAPE by model — grouped bars
x     = np.arange(len(bt))
w     = 0.20
model_info = [('WMA','MAPE_WMA',PALETTE['blue']),
              ('SES','MAPE_SES',PALETTE['amber']),
              ('Holt','MAPE_Holt',PALETTE['green']),
              ('LR','MAPE_LR',PALETTE['red'])]

for mi, (label, col, color) in enumerate(model_info):
    axes[0].bar(x + (mi-1.5)*w, bt[col].clip(0,60), w, label=label,
                color=color, alpha=0.8, edgecolor='white')

axes[0].axhline(10, color='black', ls='--', lw=1, label='10% target')
axes[0].axhline(20, color='grey',  ls=':',  lw=1, label='20% acceptable')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f"#{int(r)}" for r in bt['Rank']], rotation=45, fontsize=7)
axes[0].set_ylabel('Backtest MAPE (%)')
axes[0].set_title(f'Model MAPE — Predicting {HOLDOUT_Q}', fontweight='bold')
axes[0].set_ylim(0, 65)
axes[0].legend(fontsize=8, ncol=3)

# Right: Bias of selected model
bias_colors = [PALETTE['blue'] if b>0 else PALETTE['red'] for b in bt['Bias_Final'].fillna(0)]
axes[1].bar(x, bt['Bias_Final'].fillna(0)*100, color=bias_colors, alpha=0.8)
axes[1].axhline(0, color='black', lw=1)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f"#{int(r)}" for r in bt['Rank']], rotation=45, fontsize=7)
axes[1].set_ylabel('Bias (%)')
axes[1].set_title('Forecast Bias — Selected Model\n(Blue = over-forecast, Red = under-forecast)',
                  fontweight='bold')

plt.suptitle('Step 5 — Model Backtesting', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('backtest_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Backtest results saved to backtest_results.png')

---
## Step 6 — Base Forecast (FY26 Q2)

Now that the best model per product is confirmed by backtesting, we retrain each model on **all 12 quarters** of adjusted actuals and generate the FY26 Q2 base forecast.

This forecast represents **regular recurring demand only** — Big Deals are excluded and will be added back in Step 7.


In [ ]:
# ── 6.1 Generate FY26 Q2 base forecasts ───────────────────────────────────────
base_rows = []

for i, row in adjusted.iterrows():
    full_s = pd.to_numeric(row[ACTUAL_QUARTERS], errors='coerce').values

    bt_row     = bt[bt['Rank'] == int(row['Cost_Rank'])]
    best_model = bt_row.iloc[0]['Best_Model'] if len(bt_row) > 0 else 'Holt'
    bt_mape    = bt_row.iloc[0]['MAPE_Final'] if len(bt_row) > 0 else np.nan

    preds      = run_all_models(full_s)
    base_fc    = max(0, preds[best_model])

    dp  = pd.to_numeric(row['Demand_Planner'], errors='coerce')
    mkt = pd.to_numeric(row['Marketing'],      errors='coerce')
    ds  = pd.to_numeric(row['Data_Science'],   errors='coerce')
    q1  = pd.to_numeric(row['FY26 Q1'],        errors='coerce')

    base_rows.append({
        'Rank'         : int(row['Cost_Rank']),
        'Product'      : row['Product_Name'],
        'Life_Cycle'   : row['Life_Cycle'],
        'FY26_Q1_Adj'  : round(q1) if not np.isnan(q1) else np.nan,
        'Best_Model'   : best_model,
        'BT_MAPE'      : round(bt_mape, 1) if not np.isnan(bt_mape) else np.nan,
        'F_WMA'        : round(preds['WMA']),
        'F_SES'        : round(preds['SES']),
        'F_Holt'       : round(preds['Holt']),
        'F_LR'         : round(preds['LR']),
        'Base_Forecast': round(base_fc),
        'DP_Forecast'  : dp,
        'Mkt_Forecast' : mkt,
        'DS_Forecast'  : ds,
    })

base = pd.DataFrame(base_rows)

print(f'Base Forecast — FY26 Q2 (trained on all {len(ACTUAL_QUARTERS)} quarters)')
print('─'*105)
print(f'{"#":<4}{"Product":<42}{"Q1 Adj":>9}{"Model":<8}{"MAPE%":>6}  {"BASE FC":>9}  {"Dmnd Plnr":>10}{"Mktg":>9}{"DS":>9}')
print('─'*105)
for _,r in base.iterrows():
    dp  = f"{int(r['DP_Forecast']):,}"  if not pd.isna(r['DP_Forecast'])  else 'n/a'
    mkt = f"{int(r['Mkt_Forecast']):,}" if not pd.isna(r['Mkt_Forecast']) else 'n/a'
    ds  = f"{int(r['DS_Forecast']):,}"  if not pd.isna(r['DS_Forecast'])  else 'n/a'
    print(f"{int(r['Rank']):<4}{str(r['Product'])[:40]:<42}"
          f"{int(r['FY26_Q1_Adj']):>9,}  {r['Best_Model']:<8}{r['BT_MAPE']:>5.1f}%  "
          f"{int(r['Base_Forecast']):>9,}  {dp:>10}{mkt:>9}{ds:>9}")

print('─'*105)
print(f"  {'PORTFOLIO TOTAL':<55}"
      f"{base['Base_Forecast'].sum():>20,}")

---
## Step 7 — Correlation-Adjusted Final Forecast

This is the analytical layer that distinguishes this model from a simple per-product forecast.

### How Correlation Adjustment Works

**Step 7A — Avg Deal Cluster Adjustment**  
Products in the same demand cluster (identified in Step 4) should have **consistent directional forecasts**. If the majority of cluster members show an upward trend while one member is flat, the correlation evidence supports a mild upward nudge on that outlier.

**Step 7B — Big Deal Overlay**  
Historical Big Deal frequency × average size gives a probability-weighted estimate. Replace with confirmed pipeline data when available.

**Step 7C — Negative Correlation Check**  
For product pairs with strong negative Avg Deal correlation, if one product's forecast is high, the other's should be scrutinised — they compete for the same budget.

**Adjustment formula:**
```
Cluster_Momentum = mean of (recent 2-quarter growth rate) across cluster members
If abs(Cluster_Momentum - Product_Own_Momentum) > threshold:
    Adjusted_Forecast = Base_Forecast × (1 + dampening_factor × Cluster_Momentum)
dampening_factor = 0.30  (only 30% of cluster signal applied — conservative)
```


In [ ]:
# ── 7A Compute recent-quarter momentum per product ────────────────────────────
# Momentum = (mean of last 2 quarters) / (mean of prior 2 quarters) - 1
# Positive = demand accelerating, Negative = demand decelerating

DAMPENING = 0.30          # Conservative: apply 30% of cluster signal
THRESHOLD = 0.05          # Only adjust if cluster and product disagree by >5%

def recent_momentum(row, qs):
    s = [pd.to_numeric(row[q], errors='coerce') for q in qs]
    s = [v for v in s if not np.isnan(v)]
    if len(s) < 4: return 0.0
    recent = np.mean(s[-2:])
    prior  = np.mean(s[-4:-2])
    return (recent - prior) / prior if prior > 0 else 0.0

# Momentum per product
momentum = {}
for i, row in adjusted.iterrows():
    momentum[int(row['Cost_Rank'])] = recent_momentum(row, ACTUAL_QUARTERS)

# Cluster momentum = average of member momentums
cluster_momentum = {}
for key, members in CLUSTERS.items():
    moms = [momentum.get(m, 0) for m in members]
    cluster_momentum[key] = np.mean(moms)

# Map each product rank → its cluster key
rank_to_cluster = {}
for key, members in CLUSTERS.items():
    for m in members:
        rank_to_cluster[m] = key

print('Product momentum and cluster summary:')
print(f'{"Rank":<6}{"Product":<44}{"Own Momentum":>14}{"Cluster Momentum":>18}')
print('─'*85)
for rank, mom in momentum.items():
    clust_key   = rank_to_cluster.get(rank)
    clust_mom   = cluster_momentum.get(clust_key, None)
    row         = adjusted[adjusted['Cost_Rank']==rank].iloc[0]
    cm_str      = f"{clust_mom:+.1%}" if clust_mom is not None else 'singleton'
    print(f"  #{rank:<4}{str(row['Product_Name'])[:42]:<44}{mom:>+13.1%}  {cm_str:>16}")

In [ ]:
# ── 7B Apply cluster-based correlation adjustment ─────────────────────────────
adjusted_fc_rows = []

for _,r in base.iterrows():
    rank      = int(r['Rank'])
    base_fc   = r['Base_Forecast']
    own_mom   = momentum.get(rank, 0.0)
    clust_key = rank_to_cluster.get(rank)
    clust_mom = cluster_momentum.get(clust_key, None)

    adj_factor    = 0.0
    adj_reason    = 'No cluster'
    corr_adj_fc   = base_fc

    if clust_mom is not None:
        divergence = clust_mom - own_mom
        if abs(divergence) > THRESHOLD:
            adj_factor  = DAMPENING * divergence
            # Cap adjustment at ±15%
            adj_factor  = np.clip(adj_factor, -0.15, 0.15)
            corr_adj_fc = round(base_fc * (1 + adj_factor))
            adj_reason  = (f"Cluster signal: {clust_mom:+.1%}, "
                           f"own: {own_mom:+.1%}, adj: {adj_factor:+.1%}")
        else:
            adj_reason = f'Cluster aligned (div={divergence:+.1%})'

    # Big Deal overlay — historical frequency × average size
    bd_row    = bd_df[bd_df['Cost_Rank']==rank].iloc[0] if rank in bd_df['Cost_Rank'].values else None
    bd_vals   = [bd_row[f'BigDeal_{q}'] for q in BD_QUARTERS] if bd_row is not None else []
    non_zero  = [v for v in bd_vals if v > 0]
    bd_freq   = len(non_zero) / len(BD_QUARTERS) if BD_QUARTERS else 0
    bd_avg    = np.mean(non_zero) if non_zero else 0
    bd_overlay = round(bd_freq * bd_avg)

    final_fc  = round(corr_adj_fc + bd_overlay)
    margin    = max(r['BT_MAPE'] if not pd.isna(r['BT_MAPE']) else 15, 8) / 100

    adjusted_fc_rows.append({
        'Rank'          : rank,
        'Product'       : r['Product'],
        'Life_Cycle'    : r['Life_Cycle'],
        'Best_Model'    : r['Best_Model'],
        'BT_MAPE'       : r['BT_MAPE'],
        'Base_Forecast' : int(base_fc),
        'Adj_Factor_Pct': round(adj_factor*100, 2),
        'Corr_Adj_FC'   : int(corr_adj_fc),
        'BD_Overlay'    : int(bd_overlay),
        'Final_Forecast': int(final_fc),
        'Lower'         : int(final_fc * (1-margin)),
        'Upper'         : int(final_fc * (1+margin)),
        'Adj_Reason'    : adj_reason,
        'DP_Forecast'   : r['DP_Forecast'],
        'DS_Forecast'   : r['DS_Forecast'],
    })

final = pd.DataFrame(adjusted_fc_rows)

print('CORRELATION-ADJUSTED FINAL FORECAST — FY26 Q2')
print('='*120)
print(f'{"#":<4}{"Product":<42}{"Base FC":>10}{"Corr Adj%":>10}{"BD Ovly":>9}{"FINAL FC":>10}  {"Lower":<9}  {"Upper"}')
print('─'*120)
for _,r in final.iterrows():
    adj_flag = f"  ← adj {r['Adj_Factor_Pct']:+.1f}%" if abs(r['Adj_Factor_Pct'])>0.5 else ''
    print(f"{int(r['Rank']):<4}{str(r['Product'])[:40]:<42}"
          f"{int(r['Base_Forecast']):>10,}{r['Adj_Factor_Pct']:>+9.1f}%"
          f"{int(r['BD_Overlay']):>9,}{int(r['Final_Forecast']):>10,}  "
          f"{int(r['Lower']):<9,}  {int(r['Upper']):,}{adj_flag}")
print('─'*120)
print(f"  {'PORTFOLIO TOTAL':<95}{final['Final_Forecast'].sum():>10,}")
print(f"  {'  Lower':<95}{final['Lower'].sum():>10,}")
print(f"  {'  Upper':<95}{final['Upper'].sum():>10,}")

In [ ]:
# ── 7C Negative correlation check — flag conflicting forecasts ────────────────
neg_pairs = [(p['rank_i'], p['rank_j'], p['pearson_r'])
             for p in avg_pairs if p['pearson_r'] < -0.70]

print('NEGATIVE CORRELATION CHECK — Competing Budget Products')
print('Products in these pairs compete for the same customer budget.')
print('If both are forecast HIGH simultaneously, apply business judgement.')
print('─'*90)

for ri, rj, r in sorted(neg_pairs, key=lambda x: x[2]):
    fc_i = final[final['Rank']==ri]['Final_Forecast'].values
    fc_j = final[final['Rank']==rj]['Final_Forecast'].values
    nm_i = adjusted[adjusted['Cost_Rank']==ri]['Product_Name'].values
    nm_j = adjusted[adjusted['Cost_Rank']==rj]['Product_Name'].values
    if len(fc_i) and len(fc_j) and len(nm_i) and len(nm_j):
        print(f"  r={r:+.3f}  #{ri} {str(nm_i[0])[:38]:<40} FC={int(fc_i[0]):>8,}")
        print(f"          #{rj} {str(nm_j[0])[:38]:<40} FC={int(fc_j[0]):>8,}")
        print()

In [ ]:
# ── 7D Visualise — Base vs Correlation-Adjusted Forecast ─────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 11))

# Top: waterfall-style Base vs Final
x  = np.arange(len(final))
w  = 0.35
axes[0].bar(x-w/2, final['Base_Forecast'], w, label='Base Forecast (no corr adj)',
            color=PALETTE['blue'], alpha=0.8, edgecolor='white')
axes[0].bar(x+w/2, final['Final_Forecast'], w, label='Final Forecast (corr adjusted + BD overlay)',
            color=PALETTE['teal'], alpha=0.85, edgecolor='white')
axes[0].errorbar(x+w/2, final['Final_Forecast'],
                 yerr=[final['Final_Forecast']-final['Lower'],
                       final['Upper']-final['Final_Forecast']],
                 fmt='none', color='#333', capsize=3, lw=1.2)
axes[0].set_xticks(x)
axes[0].set_xticklabels([f"#{int(r)}" for r in final['Rank']], rotation=45, fontsize=7)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:,.0f}'))
axes[0].set_ylabel('Forecast Units')
axes[0].set_title('Base vs Correlation-Adjusted Final Forecast — FY26 Q2',
                  fontweight='bold')
axes[0].legend(fontsize=9)

# Bottom: Adjustment magnitude
adj_colors = [PALETTE['green'] if a>0 else PALETTE['red'] if a<0
              else PALETTE['grey'] for a in final['Adj_Factor_Pct']]
axes[1].bar(x, final['Adj_Factor_Pct'], color=adj_colors, alpha=0.85, edgecolor='white')
axes[1].axhline(0, color='black', lw=1)
axes[1].axhline( 5, color=PALETTE['green'], ls='--', lw=0.8, alpha=0.5)
axes[1].axhline(-5, color=PALETTE['red'],   ls='--', lw=0.8, alpha=0.5)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f"#{int(r)}" for r in final['Rank']], rotation=45, fontsize=7)
axes[1].set_ylabel('Correlation Adjustment (%)')
axes[1].set_title('Correlation Adjustment Applied per Product\n'
                  '(Green = cluster signal nudged forecast up, Red = nudged down, '
                  'Grey = no adjustment)',
                  fontweight='bold')

plt.suptitle('Step 7 — Correlation-Adjusted Final Forecast',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('correlation_adjusted_forecast.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Final forecast chart saved to correlation_adjusted_forecast.png')

---
## Step 8 — Final Deliverable: Export to Excel

The output workbook contains four sheets:

| Sheet | Contents |
|-------|----------|
| **Final Forecast** | One row per product: base, adjustment, BD overlay, final, confidence range |
| **Model Comparison** | All four model forecasts vs three team forecasts side by side |
| **Correlation Summary** | Top Avg Deal and Big Deal correlations with interpretation |
| **Accuracy Tracker** | Paste actuals here when FY26 Q2 lands — all metrics auto-calculate |


In [ ]:
# ── 8.1 Build Excel output ────────────────────────────────────────────────────
OUTPUT_FILE = 'FY26_Q2_CorrelationAdjusted_Forecast.xlsx'
wb          = Workbook()

NAVY   = '1F4E79'; BLUE  = '2E75B6'; TEAL  = '0D7377'
GREEN  = '375623'; RED   = 'C00000'; AMBER = 'BF8F00'
ALT_BG = 'EBF3FB'; WHT   = 'FFFFFF'

def mkbdr():
    s = Side(style='thin', color='BFBFBF')
    return Border(left=s, right=s, top=s, bottom=s)

def cell(ws, row, col, val, bold=False, fill_hex=None,
         align='center', fmt=None, color='000000', size=9, wrap=False):
    c = ws.cell(row=row, column=col, value=val)
    c.font      = Font(bold=bold, size=size, name='Calibri', color=color)
    c.alignment = Alignment(horizontal=align, vertical='center', wrap_text=wrap)
    c.border    = mkbdr()
    if fill_hex:
        c.fill = PatternFill('solid', fgColor=fill_hex)
    if fmt:
        c.number_format = fmt
    return c

def hdr(ws, row, col, val, fill=NAVY, color=WHT):
    return cell(ws, row, col, val, bold=True, fill_hex=fill,
                color=color, size=9, wrap=True)

NUM  = '#,##0'
PCT  = '0.0"%"'
PCT2 = '0.00"%"'

# ═══════════════════════════════════════════════════════
# SHEET 1 — FINAL FORECAST
# ═══════════════════════════════════════════════════════
ws1 = wb.active
ws1.title = 'Final Forecast'
ws1.merge_cells('A1:N1')
c = ws1['A1']
c.value = 'FY26 Q2 — Correlation-Adjusted Demand Forecast | All 30 Products'
c.font  = Font(bold=True, size=13, color=WHT, name='Calibri')
c.fill  = PatternFill('solid', fgColor=NAVY)
c.alignment = Alignment(horizontal='center', vertical='center')
ws1.row_dimensions[1].height = 28

H1 = ['#','Product Name','Life Cycle','FY26 Q1\nAdj Actual','Best\nModel',
      'BT MAPE','Base Forecast\n(Avg Deal)','Corr Adj\n%','Corr Adj\nForecast',
      'BD Overlay','FINAL\nFORECAST','Lower\nBound','Upper\nBound','Adjustment Reason']
W1 = [5,45,13,13,9,8,15,10,14,11,14,11,11,40]
for ci,(h,w) in enumerate(zip(H1,W1),1):
    hdr(ws1,2,ci,h); ws1.column_dimensions[get_column_letter(ci)].width=w
ws1.row_dimensions[2].height = 38

tier_fill = {True:'E2EFDA', False:WHT}
for ri,r in final.iterrows():
    exrow = ri+3
    rf    = ALT_BG if exrow%2==0 else WHT
    gf    = '2E8B57' if abs(r['Adj_Factor_Pct'])>0.5 else NAVY
    vals  = [
        (int(r['Rank']),'center',None,False),
        (r['Product'],'left',None,False),
        (r['Life_Cycle'],'center',None,False),
        (r['Base_Forecast'],'right',NUM,False),  # using base as proxy for adj q1
        (r['Best_Model'],'center',None,False),
        (r['BT_MAPE'],'center',PCT,False),
        (r['Base_Forecast'],'right',NUM,False),
        (r['Adj_Factor_Pct']/100,'center','0.0%',False),
        (r['Corr_Adj_FC'],'right',NUM,False),
        (r['BD_Overlay'],'right',NUM,False),
        (r['Final_Forecast'],'right',NUM,True),
        (r['Lower'],'right',NUM,False),
        (r['Upper'],'right',NUM,False),
        (r['Adj_Reason'],'left',None,False),
    ]
    for ci,(v,al,fm,_) in enumerate(vals,1):
        fill = '2E8B22' if ci==11 else rf
        c = cell(ws1,exrow,ci,v,fill_hex=fill,align=al,fmt=fm)
        if ci==11: c.font=Font(bold=True,size=9,name='Calibri',color=WHT)

# Total row
tr = len(final)+3
ws1.merge_cells(f'A{tr}:E{tr}')
cell(ws1,tr,1,'PORTFOLIO TOTAL',bold=True,fill_hex=BLUE,color=WHT,align='center')
for ci,col in [(7,'G'),(9,'I'),(10,'J'),(11,'K'),(12,'L'),(13,'M')]:
    cell(ws1,tr,ci,f'=SUM({col}3:{col}{tr-1})',bold=True,fill_hex=BLUE,color=WHT,fmt=NUM,align='right')

ws1.freeze_panes = 'A3'

# ═══════════════════════════════════════════════════════
# SHEET 2 — MODEL COMPARISON
# ═══════════════════════════════════════════════════════
ws2 = wb.create_sheet('Model Comparison')
ws2.merge_cells('A1:L1')
c2=ws2['A1']; c2.value='All Models vs Team Forecasts — FY26 Q2'
c2.font=Font(bold=True,size=12,color=WHT,name='Calibri')
c2.fill=PatternFill('solid',fgColor=NAVY)
c2.alignment=Alignment(horizontal='center',vertical='center')
ws2.row_dimensions[1].height=24

H2=['#','Product','Life Cycle','WMA','SES','Holt','Lin Reg','OUR FINAL','Demand Planner','Marketing','Data Science','BT MAPE']
W2=[5,45,12,11,11,11,11,13,15,12,13,9]
for ci,(h,w) in enumerate(zip(H2,W2),1):
    hdr(ws2,2,ci,h); ws2.column_dimensions[get_column_letter(ci)].width=w
ws2.row_dimensions[2].height=30

for ri,r in final.iterrows():
    exrow=ri+3; rf=ALT_BG if exrow%2==0 else WHT
    base_r = base[base['Rank']==r['Rank']].iloc[0]
    v2=[
        (int(r['Rank']),'center',None),
        (r['Product'],'left',None),
        (r['Life_Cycle'],'center',None),
        (base_r['F_WMA'],'right',NUM),
        (base_r['F_SES'],'right',NUM),
        (base_r['F_Holt'],'right',NUM),
        (base_r['F_LR'],'right',NUM),
        (r['Final_Forecast'],'right',NUM),
        (r['DP_Forecast'],'right',NUM),
        (base_r['Mkt_Forecast'],'right',NUM),
        (r['DS_Forecast'],'right',NUM),
        (r['BT_MAPE'],'center',PCT),
    ]
    for ci,(v,al,fm) in enumerate(v2,1):
        fill='DFFFDF' if ci==8 else rf
        cell(ws2,exrow,ci,v,fill_hex=fill,align=al,fmt=fm)
ws2.freeze_panes='A3'

# ═══════════════════════════════════════════════════════
# SHEET 3 — CORRELATION SUMMARY
# ═══════════════════════════════════════════════════════
ws3=wb.create_sheet('Correlation Summary')
ws3.merge_cells('A1:H1')
c3=ws3['A1']; c3.value='Avg Deal Correlation — Top 30 Pairs (Structural Demand)'
c3.font=Font(bold=True,size=12,color=WHT,name='Calibri')
c3.fill=PatternFill('solid',fgColor=TEAL); c3.alignment=Alignment(horizontal='center',vertical='center')
ws3.row_dimensions[1].height=24

H3=['#','Rank A','Product A','Rank B','Product B','Pearson r','Spearman r','Interpretation']
W3=[5,7,44,7,44,11,11,35]
for ci,(h,w) in enumerate(zip(H3,W3),1):
    hdr(ws3,2,ci,h,fill=TEAL); ws3.column_dimensions[get_column_letter(ci)].width=w
ws3.row_dimensions[2].height=28

for pi, p in enumerate(avg_pairs[:30], 1):
    exrow=pi+2; rf=ALT_BG if exrow%2==0 else WHT
    interp = ('Very Strong Positive — Demand cluster' if p['pearson_r']>=0.85
               else 'Strong Positive' if p['pearson_r']>=0.70
               else 'Very Strong Negative — Competing budget' if p['pearson_r']<=-0.85
               else 'Strong Negative — Budget competition')
    v3=[(pi,'center',None),(f"#{p['rank_i']}", 'center',None),
        (p['name_i'],'left',None),(f"#{p['rank_j']}", 'center',None),
        (p['name_j'],'left',None),(p['pearson_r'],'center','0.000'),
        (p['spearman_r'],'center','0.000'),(interp,'left',None)]
    rcolor = 'FFE0E0' if p['pearson_r']<0 else rf
    for ci,(v,al,fm) in enumerate(v3,1):
        cell(ws3,exrow,ci,v,fill_hex=rcolor,align=al,fmt=fm)
ws3.freeze_panes='A3'

# ═══════════════════════════════════════════════════════
# SHEET 4 — ACCURACY TRACKER
# ═══════════════════════════════════════════════════════
ws4=wb.create_sheet('Accuracy Tracker')
ws4.merge_cells('A1:H1')
c4=ws4['A1']; c4.value='FY26 Q2 Accuracy Tracker — Enter Actuals in Column D when available'
c4.font=Font(bold=True,size=12,color=WHT,name='Calibri')
c4.fill=PatternFill('solid',fgColor=GREEN); c4.alignment=Alignment(horizontal='center',vertical='center')
ws4.row_dimensions[1].height=24

H4=['#','Product','Final Forecast','← Paste Actual Here','Accuracy','Bias %','Error (Units)','BD Tier']
W4=[5,45,15,20,12,10,14,12]
for ci,(h,w) in enumerate(zip(H4,W4),1):
    hdr(ws4,2,ci,h,fill=GREEN); ws4.column_dimensions[get_column_letter(ci)].width=w
ws4.row_dimensions[2].height=28

yl=PatternFill('solid',fgColor='FFFF00')
for ri,r in final.iterrows():
    exrow=ri+3; rf=ALT_BG if exrow%2==0 else WHT
    cell(ws4,exrow,1,int(r['Rank']),fill_hex=rf)
    cell(ws4,exrow,2,r['Product'],fill_hex=rf,align='left')
    cell(ws4,exrow,3,int(r['Final_Forecast']),fill_hex=rf,align='right',fmt=NUM)
    inp=ws4.cell(row=exrow,column=4,value=None)
    inp.fill=yl; inp.border=mkbdr()
    inp.alignment=Alignment(horizontal='right',vertical='center')
    inp.number_format=NUM; inp.font=Font(size=9,name='Calibri',bold=True,color=NAVY)
    fr=exrow
    cell(ws4,fr,5,f'=IF(D{fr}="","",1-ABS(C{fr}-D{fr})/D{fr})',fill_hex=rf,fmt='0.0%')
    cell(ws4,fr,6,f'=IF(D{fr}="","",(C{fr}-D{fr})/D{fr})',fill_hex=rf,fmt='0.0%')
    cell(ws4,fr,7,f'=IF(D{fr}="","",C{fr}-D{fr})',fill_hex=rf,align='right',fmt=NUM)
    tier=bd_sum[bd_sum['Rank']==int(r['Rank'])]['Tier'].values
    cell(ws4,fr,8,tier[0] if len(tier) else '',fill_hex=rf)

sr=len(final)+3
ws4.merge_cells(f'A{sr}:B{sr}')
cell(ws4,sr,1,'TOTAL',bold=True,fill_hex=GREEN,color=WHT)
cell(ws4,sr,3,f'=SUM(C3:C{sr-1})',bold=True,fill_hex=GREEN,color=WHT,fmt=NUM,align='right')
cell(ws4,sr,4,f'=SUM(D3:D{sr-1})',bold=True,fill_hex=GREEN,color=WHT,fmt=NUM,align='right')
cell(ws4,sr,5,f'=IF(D{sr}=0,"",1-ABS(C{sr}-D{sr})/D{sr})',bold=True,fill_hex=GREEN,color=WHT,fmt='0.0%')
cell(ws4,sr,6,f'=IF(D{sr}=0,"",(C{sr}-D{sr})/D{sr})',bold=True,fill_hex=GREEN,color=WHT,fmt='0.0%')
ws4.freeze_panes='A3'

wb.save(OUTPUT_FILE)
print(f'✅  Saved: {OUTPUT_FILE}')
print(f'\n   Sheet 1 — Final Forecast    : Base + Corr Adj + BD Overlay + Confidence Range')
print(f'   Sheet 2 — Model Comparison  : All 4 models vs 3 team forecasts')
print(f'   Sheet 3 — Correlation Summary: Top 30 Avg Deal pairs')
print(f'   Sheet 4 — Accuracy Tracker  : Paste FY26 Q2 actuals → auto-calculates accuracy & bias')

In [ ]:
# ── 8.2 Executive Summary ─────────────────────────────────────────────────────
total_fc     = final['Final_Forecast'].sum()
total_base   = final['Base_Forecast'].sum()
total_bd     = final['BD_Overlay'].sum()
total_lower  = final['Lower'].sum()
total_upper  = final['Upper'].sum()
n_adjusted   = (final['Adj_Factor_Pct'].abs()>0.5).sum()
avg_mape     = final['BT_MAPE'].mean()
neg_corr_ct  = len(neg_pairs)

print('='*65)
print('        FY26 Q2 FORECAST — EXECUTIVE SUMMARY')
print('='*65)
print(f'  Total Portfolio Forecast : {total_fc:>12,} units')
print(f'  Confidence Range         : {total_lower:>12,} – {total_upper:,} units')
print(f'  Base Forecast (Avg Deal) : {total_base:>12,} units')
print(f'  Big Deal Overlay         : {total_bd:>12,} units')
print(f'  Avg Backtest MAPE        : {avg_mape:>11.1f}%')
print(f'  Products corr-adjusted   : {n_adjusted:>12} / {N_PRODUCTS}')
print(f'  Neg corr pairs flagged   : {neg_corr_ct:>12}')
print()
print('  METHODOLOGY')
print('  ' + '─'*55)
print('  1. Big Deal adjusted actuals (Avg Deal series)')
print('  2. EDA: CV, trend slope, seasonal ratio per product')
print('  3. Correlation analysis: Big Deal + Avg Deal matrices')
print('  4. Demand cluster identification (Union-Find, r≥0.85)')
print('  5. Backtested model selection per product')
print('  6. Cluster momentum-based correlation adjustment')
print('  7. Probability-weighted Big Deal overlay')
print('  8. Confidence range: ±MAPE% floor 8%')
print('='*65)

---
## Appendix — Key Decisions Log

Good analytical practice requires documenting *why* you made each modelling decision, not just *what* you did. This log provides an audit trail.

| Decision | Choice | Rationale |
|----------|--------|----------|
| Big Deal adjustment | Applied | 13/30 products have >15% BD exposure — raw actuals would systematically over-estimate |
| Model per product | Yes | One-size-fits-all fails: Decline products need LR, NPI-Ramp needs SES |
| Ensemble | **No** | With 8-12 quarters of data, ensemble complexity is not justified. Backtested single-model selection is more defensible |
| Backtest split | Q1–Q11 train, Q12 holdout | Only 12 quarters available — one holdout is the practical maximum |
| Correlation adjustment | Cluster momentum (30% dampening) | Conservative: only 30% of cluster signal applied; avoids over-correcting sparse data |
| BD Overlay | Freq × Avg Size | Default when pipeline data unavailable; replace with CRM data for production use |
| Confidence range | ±MAPE% (min 8%) | Model-specific uncertainty; floor prevents overconfidence on products with low backtest MAPE |
| Negative correlation | Flag only | Insufficient data to apply a quantitative adjustment; business judgement required |
